<a href="https://colab.research.google.com/github/100522128/g81_P2_03_Aprendizaje_Automatico/blob/main/notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Práctica 2 - Determinacion Grupos de Estrellas
**Miembros del grupo:**

Christian Cano -> 100522320

Adrián Curell -> 100522128

[Repositorio de github](https://github.com/100522128/g81_P2_03_Aprendizaje_Automatico.git)

##1 Visualizacion de los datos

###1.1 Carga del dataset

Cargamos el archivo “starts_data.csv" usando pandas

In [ ]:
import pandas as pd
from google.colab import files
archivos = files.upload()
df=pd.read_csv("stars_data.csv")

Saving stars_data.csv to stars_data (3).csv


###1.2 Tipos de variables

Ahora clasificamos las variables segun su tipo

In [ ]:
print(df.dtypes.values)

[dtype('int64') dtype('float64') dtype('float64') dtype('float64')
 dtype('O') dtype('O')]


Como podemos ver todas son o int64 o Object por lo que podemos hacer un clasificador automático que las separe en numéricas (int64) o no numéricas (Object) que se dividirá en categóricas o ordinales. Para realizarlo haremos:

1. Dividir las variables en numéricas o no numéricas según su dtype utilizando la función select_dtypes.

2. De las no numéricas sacamos con unique los distintos valores que tienen y con nunique el numero total.

  

3. Creamos una lista para el tipo (numéricas o ordinal)

4. Recorrer el dataframe con un bucle y rellenando estas listas con los datos de las variables.

5. Crear un dataframe donde introduzcamos los datos de estas listas juntos al nombre y dtype de cada variable.

In [ ]:
import numpy as np

# Clasificación automática de variables
variables_numericas = df.select_dtypes(include=[np.number]).columns.tolist()
variables_no_numericas = df.select_dtypes(include=['object']).columns.tolist()

N_unique = []
Valores = []

for i in variables_no_numericas:
  N_unique.append(df[i].nunique())
  Valores.append(df[i].unique())

cardinalidad = pd.DataFrame({
    'Variable': variables_no_numericas,
    'N_unique': N_unique,
    'Valores': Valores
})
print(cardinalidad[['Variable', 'N_unique', 'Valores']].to_string(index=False))

variables_categoricas = []
variables_ordinales = []

for x in variables_no_numericas:
    variables_ordinales.append(x)



# Imprimimos la clasificación
print('\n=== Variables NUMÉRICAS ===')
print(variables_numericas)

print('\n=== Variables ORDINALES ===')
print(variables_ordinales)

# Creamos listas vacías donde almacenar los tipos y roles de cada variable
lista_tipo = []

# Recorrer columnas
for c in df.columns:
  # Tipo (Numérica, Categórica o Ordinal)
  if c in variables_numericas:
    lista_tipo.append('Numérica')
  elif c in variables_categoricas:
      lista_tipo.append('Categórica')
  else:
    lista_tipo.append('Ordinal')



# Crear DataFrame final
resumen_tipos = pd.DataFrame({
    'Variable': df.columns,
    'Dtype': df.dtypes.values,
    'Tipo': lista_tipo,
})

print('\n', resumen_tipos.to_string(index=True))

      Variable  N_unique                                                                                                                                                                                       Valores
         Color        17 [Red, Blue White, White, Yellowish White, Blue white, Pale yellow orange, Blue, Blue-white, Whitish, yellow-white, Orange, White-Yellow, white, yellowish, Yellowish, Orange-Red, Blue-White]
Spectral_Class         7                                                                                                                                                                         [M, B, A, F, O, K, G]

=== Variables NUMÉRICAS ===
['Temperature', 'L', 'R', 'A_M']

=== Variables ORDINALES ===
['Color', 'Spectral_Class']

          Variable    Dtype      Tipo
0     Temperature    int64  Numérica
1               L  float64  Numérica
2               R  float64  Numérica
3             A_M  float64  Numérica
4           Color   object   Ordinal
5  Spectral_

Vemos que son variables ordinales Color y Spectral_Class, siendo el resto variables numericas

###1.3 Valores faltantes

Ahora comprobamos si hay valores faltantes en alguna variable

In [ ]:
faltan = pd.DataFrame({
    'Variable': df.columns,
    'Valores faltantes': df.isnull().sum().values,
    '% faltantes': (df.isnull().mean() * 100).round(2).values
})
faltan = faltan[faltan['Valores faltantes'] > 0]

if faltan.empty:
    print('No hay valores faltantes (NaN) en el dataset.')
else:
    print('Variables con valores faltantes:')
    print(faltan.to_string(index=False))

No hay valores faltantes (NaN) en el dataset.


Tras la comprobacion se puede afirmar que no existen valores faltantes por lo que podemos pasar al tratamiento de las variables ordinales directamente

###1.4 Conversión de variables ordinales

En esta etapa del preprocesado, se procedió a la transformación de las variables cualitativas Color y Spectral_Class en magnitudes numéricas mediante un mapeo ordinal. Esta decisión técnica se fundamenta en que ambas variables poseen una jerarquía física intrínseca vinculada a la termodinámica estelar.


Variable Color: Se estableció una escala numérica donde el valor mínimo ($0$) se asignó a las longitudes de onda asociadas a menores temperaturas (colores cálidos como el rojo) y el valor máximo a las estrellas de mayor temperatura superficial (colores de alta energía como el azul).

Variable Spectral_Class: Se aplicó una lógica análoga siguiendo estrictamente la secuencia térmica proporcionada en las directrices de la práctica: (O, B, A, F, G, K, M). En este modelo, las estrellas de tipo M (más frías) representan el extremo inferior de la escala, mientras que las de tipo O (más calientes) definen el extremo superior, preservando así la relación de energía necesaria para el análisis de clustering posterior.

Esta codificación permite que los algoritmos de aprendizaje no supervisado interpreten correctamente la proximidad física entre los diferentes tipos de estrellas, factor que se perdería con otras técnicas como el one-hot encoding.

In [ ]:
# 1. Configuración inicial
SEMILLA = 100522320
np.random.seed(SEMILLA)

# 2. Limpieza de la variable Color
# Convertimos a minúsculas y eliminamos espacios/guiones para unificar
df['Color'] = df['Color'].str.lower().str.replace('-', ' ').str.strip()

# 3. Diccionarios de mapeo (Criterio: Mayor número = Mayor temperatura/energía)
# Mapeo Espectral
spectral_map = {'o': 6, 'b': 5, 'a': 4, 'f': 3, 'g': 2, 'k': 1, 'm': 0}

# Mapeo de Color (Basado en la secuencia térmica estelar)
color_map = {
    'red': 0,
    'orange red': 1,
    'orange': 2,
    'pale yellow orange': 3,
    'yellowish': 4,
    'yellowish white': 5,
    'yellow white': 5,
    'white yellow': 5,
    'whitish': 6,
    'white': 7,
    'blue white': 8,
    'blue': 9
}

#aui
df['Spectral_Class_num'] = df['Spectral_Class'].str.lower().map(spectral_map)
df['Color_num'] = df['Color'].map(color_map)

#Variables a procesar
features_to_model = ['Temperature', 'L', 'R', 'A_M', 'Spectral_Class_num', 'Color_num']
X = df[features_to_model].copy()

#Comprobacion
transformacion1=df['Color_num'].unique()
transformacion2=df['Spectral_Class_num'].unique()
print(transformacion1)
print(transformacion2)


[0 8 7 5 3 9 6 2 4 1]
[0 5 4 3 6 1 2]


Como podemos ver hemos creado dos nuevas columnas color_num y Spectral_Class_num que representan la codificación de las variables ordinales